# 🧹 Data Lake Cleanup

**Run this notebook in Google Colab with Google Drive mounted.**

Operations:
1. Migrate legacy `01_bronze/` datasets to categorized bronze folders
2. Remove 14 empty stub datasets from bronze
3. Clean 49 promoted datasets from `00_landing/kaggle/`
4. Remove legacy `01_bronze/` after migration
5. Regenerate `MANIFEST.json`

In [1]:
# Cell 1: Mount Drive
from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
# Cell 2: Setup
import shutil
from pathlib import Path

DATA_LAKE = Path('/content/drive/MyDrive/data_lake')

def safe_remove(path: Path, dry_run=False):
    """Remove a directory or file with logging."""
    if path.exists():
        if dry_run:
            print(f'  [DRY RUN] Would remove: {path.name}')
        else:
            shutil.rmtree(path)
            print(f'  ✅ Removed: {path.name}')
    else:
        print(f'  ⏭️  Already gone: {path.name}')

def safe_move(src: Path, dst_parent: Path, dry_run=False):
    """Move a directory to a new parent."""
    dst = dst_parent / src.name
    if not src.exists():
        print(f'  ⏭️  Source missing: {src}')
        return
    if dst.exists():
        print(f'  ⚠️  Already exists at destination: {dst.name} (skipping, remove source manually)')
        return
    if dry_run:
        print(f'  [DRY RUN] Would move: {src.name} → {dst_parent.name}/')
    else:
        shutil.move(str(src), str(dst))
        print(f'  ✅ Moved: {src.name} → {dst_parent.name}/')

# Set to False to actually execute
DRY_RUN = True
print(f'Mode: {"DRY RUN (preview only)" if DRY_RUN else "🔴 LIVE — changes will be made"}')

Mode: DRY RUN (preview only)


In [3]:
# Cell 3: Step 1 — Migrate legacy 01_bronze/
print('=== STEP 1: Migrate legacy 01_bronze/ ===')
legacy_bronze = DATA_LAKE / '01_bronze'

if legacy_bronze.exists():
    for d in sorted(legacy_bronze.iterdir()):
        if not d.is_dir():
            continue
        if d.name == 'cifar10_dataset':
            safe_move(d, DATA_LAKE / '01_bronze_vision', DRY_RUN)
        elif d.name == 'medical':
            # medical/ contains breast-ultrasound which already exists in 01_bronze_medical
            # Just remove the duplicate
            print(f'  medical/ contains: {[x.name for x in d.iterdir()]}')
            safe_remove(d, DRY_RUN)
        else:
            print(f'  ❓ Unexpected: {d.name}')

    # Remove legacy 01_bronze if empty
    if not DRY_RUN and legacy_bronze.exists() and not any(legacy_bronze.iterdir()):
        legacy_bronze.rmdir()
        print('  ✅ Removed empty 01_bronze/')
else:
    print('  ⏭️  01_bronze/ already removed')

=== STEP 1: Migrate legacy 01_bronze/ ===
  ⏭️  01_bronze/ already removed


In [4]:
# Cell 4: Step 2 — Remove 14 empty stub datasets
print('=== STEP 2: Remove empty stub datasets ===')

EMPTY_DATASETS = [
    '01_bronze_audio/audioset',
    '01_bronze_medical/zenodo-bus-bra#',
    '01_bronze_nlp/conll2003',
    '01_bronze_nlp/quora-pairs',
    '01_bronze_nlp/toxic-comments',
    '01_bronze_tabular/house-prices',
    '01_bronze_tabular/predict-future-sales',
    '01_bronze_tabular/rossmann',
    '01_bronze_tabular/spaceship-titanic',
    '01_bronze_tabular/walmart-sales',
    '01_bronze_timeseries/electricity-demand',
    '01_bronze_timeseries/m4-hourly',
    '01_bronze_vision/fer-plus',
    '01_bronze_vision/fer2013',
]

for rel_path in EMPTY_DATASETS:
    full_path = DATA_LAKE / rel_path
    safe_remove(full_path, DRY_RUN)

print(f'\n{len(EMPTY_DATASETS)} empty stubs targeted')

=== STEP 2: Remove empty stub datasets ===
  ⏭️  Already gone: audioset
  ⏭️  Already gone: zenodo-bus-bra#
  ⏭️  Already gone: conll2003
  ⏭️  Already gone: quora-pairs
  ⏭️  Already gone: toxic-comments
  ⏭️  Already gone: house-prices
  ⏭️  Already gone: predict-future-sales
  ⏭️  Already gone: rossmann
  ⏭️  Already gone: spaceship-titanic
  ⏭️  Already gone: walmart-sales
  ⏭️  Already gone: electricity-demand
  ⏭️  Already gone: m4-hourly
  ⏭️  Already gone: fer-plus
  ⏭️  Already gone: fer2013

14 empty stubs targeted


In [5]:
# Cell 5: Step 3 — Clean promoted datasets from 00_landing/kaggle/
print('=== STEP 3: Clean promoted landing data ===')

PROMOTED_LANDING = [
    'animals10', 'blood-cells', 'bone-fracture', 'brain-tumor',
    'breast-ultrasound', 'bus-uclm-breast', 'butterflies',
    'chest-ct', 'chest-xray', 'covid-ct', 'covid-time-series',
    'credit-fraud', 'crypto-prices', 'dental-xray',
    'electricity-demand', 'emotion-speech', 'energy-consumption',
    'eurosat', 'fake-news', 'fer2013', 'flowers', 'fruits360',
    'garbage', 'gender-recognition', 'gtzan', 'heartbeat',
    'house-prices', 'intel-image', 'landscape', 'leukemia',
    'lung-cancer', 'lung-ct', 'm4-hourly', 'malaria', 'mura-xray',
    'natural-images', 'predict-future-sales', 'quora-pairs',
    'rice-images', 'rossmann', 'sign-language', 'skin-cancer',
    'spaceship-titanic', 'spam', 'stock-market', 'toxic-comments',
    'twitter-sentiment', 'walmart-sales', 'weather',
]

landing_kaggle = DATA_LAKE / '00_landing' / 'kaggle'
for name in PROMOTED_LANDING:
    safe_remove(landing_kaggle / name, DRY_RUN)

print(f'\n{len(PROMOTED_LANDING)} promoted landing folders targeted')

=== STEP 3: Clean promoted landing data ===
  ⏭️  Already gone: animals10
  ⏭️  Already gone: blood-cells
  ⏭️  Already gone: bone-fracture
  ⏭️  Already gone: brain-tumor
  ⏭️  Already gone: breast-ultrasound
  ⏭️  Already gone: bus-uclm-breast
  ⏭️  Already gone: butterflies
  ⏭️  Already gone: chest-ct
  ⏭️  Already gone: chest-xray
  ⏭️  Already gone: covid-ct
  ⏭️  Already gone: covid-time-series
  ⏭️  Already gone: credit-fraud
  ⏭️  Already gone: crypto-prices
  ⏭️  Already gone: dental-xray
  ⏭️  Already gone: electricity-demand
  ⏭️  Already gone: emotion-speech
  ⏭️  Already gone: energy-consumption
  ⏭️  Already gone: eurosat
  ⏭️  Already gone: fake-news
  ⏭️  Already gone: fer2013
  ⏭️  Already gone: flowers
  ⏭️  Already gone: fruits360
  ⏭️  Already gone: garbage
  ⏭️  Already gone: gender-recognition
  ⏭️  Already gone: gtzan
  ⏭️  Already gone: heartbeat
  ⏭️  Already gone: house-prices
  ⏭️  Already gone: intel-image
  ⏭️  Already gone: landscape
  ⏭️  Already gone: l

In [6]:
# Cell 6: Step 4 — Also clean other landing source folders if empty
print('=== STEP 4: Clean empty landing source folders ===')

landing = DATA_LAKE / '00_landing'
for source_dir in sorted(landing.iterdir()):
    if not source_dir.is_dir():
        continue
    if source_dir.name == 'kaggle':
        continue  # Handled above
    children = list(source_dir.iterdir())
    if len(children) == 0:
        safe_remove(source_dir, DRY_RUN)
    else:
        print(f'  Kept: {source_dir.name}/ ({len(children)} items)')

=== STEP 4: Clean empty landing source folders ===
  Kept: archive/ (17 items)
  Kept: breast-ultrasound/ (1 items)
  Kept: huggingface/ (31 items)
  Kept: openml/ (6 items)
  Kept: polyp/ (12 items)
  Kept: shared/ (16 items)
  Kept: shared_thinh/ (3 items)
  Kept: sklearn/ (5 items)
  Kept: tfds/ (10 items)
  Kept: torchvision/ (10 items)
  Kept: uci/ (19 items)
  Kept: url/ (15 items)


In [7]:
# Cell 7: Summary — Show what's left
print('=== CURRENT STATE ===')
for d in sorted(DATA_LAKE.iterdir()):
    if d.is_dir():
        try:
            count = sum(1 for _ in d.iterdir())
        except:
            count = '?'
        print(f'  {d.name}/  ({count} items)')
    else:
        print(f'  {d.name}  ({d.stat().st_size} bytes)')

print(f'\n⚠️  Mode was: {"DRY RUN" if DRY_RUN else "LIVE"}')
if DRY_RUN:
    print('Set DRY_RUN = False in Cell 2 and re-run to execute changes.')

=== CURRENT STATE ===
  00_landing/  (13 items)
  01_bronze_audio/  (5 items)
  01_bronze_detection/  (3 items)
  01_bronze_education/  (3 items)
  01_bronze_generative/  (1 items)
  01_bronze_medical/  (24 items)
  01_bronze_nlp/  (18 items)
  01_bronze_tabular/  (19 items)
  01_bronze_timeseries/  (5 items)
  01_bronze_video/  (0 items)
  01_bronze_vision/  (45 items)
  02_silver/  (1 items)
  03_gold/  (0 items)
  04_platinum/  (0 items)
  05_features/  (0 items)
  06_inference/  (1 items)
  09_cache/  (0 items)
  MANIFEST.json  (35937 bytes)

⚠️  Mode was: DRY RUN
Set DRY_RUN = False in Cell 2 and re-run to execute changes.


In [8]:
# Cell 8: Regenerate MANIFEST.json (run AFTER live execution)
# Only run this cell after DRY_RUN = False has been executed

if not DRY_RUN:
    from shared_config.manifest import generate_manifest
    m = generate_manifest()
    print(f"\n📋 MANIFEST regenerated:")
    print(f"   Datasets: {m['summary']['total_datasets']}")
    print(f"   Files: {m['summary']['total_files']}")
    print(f"   Size: {m['summary']['total_size_gb']} GB")
    print(f"   Empty: {m['summary']['empty_datasets']}")
else:
    print('Skipping MANIFEST regeneration (DRY_RUN mode)')

Skipping MANIFEST regeneration (DRY_RUN mode)
